# Multivariate forecasting example in Python

In this notebook we use a time series with multiple input time series, i.e. the various price points of a stock during the day, to predict a single output time series which is the opening price the next day. We use the real-life data of the AU Small Finance Bank between the beginning and middle of 2021 in the corresponding 'AUBANK.NS.csv' file. We will use XGBoost, as well as convert the date information into relevant features.

First we import the required packages:

In [ ]:
import plotly.express as px
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
df = pd.read_csv('AUBANK.NS.csv')
df

,Date,Open,High,Low,Close,Adj Close,Volume
0,2021-01-11,900.849976,904.500000,873.500000,899.450012,899.450012,564393
1,2021-01-12,895.500000,908.000000,894.950012,904.849976,904.849976,472697
2,2021-01-13,908.950012,908.950012,884.150024,892.500000,892.500000,730301
3,2021-01-14,896.400024,931.500000,896.400024,914.000000,914.000000,1015700
4,2021-01-15,914.000000,921.900024,907.250000,920.250000,920.250000,1411827
...,...,...,...,...,...,...,...
117,2021-07-02,1024.000000,1042.199951,1024.000000,1038.849976,1038.849976,726010
118,2021-07-05,1044.949951,1047.500000,1021.750000,1030.349976,1030.349976,656418
119,2021-07-06,1056.000000,1142.750000,1056.000000,1114.300049,1114.300049,8577690
120,2021-07-07,1109.000000,1133.000000,1091.000000,1125.000000,1125.000000,1542754


Let's first plot the time series for variable 'Open':

In [ ]:
fig = px.line(df, x='Date', y="Open")
fig.show()

Let's also convert the 'Date' column into an actual date:

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df.dtypes

,0
Date,datetime64[ns]
Open,float64
High,float64
Low,float64
Close,float64
Adj Close,float64
Volume,int64


Given that we deal with the price as well as the volume, some scaling is in order. Standard normal scaling is used for all variables:

In [ ]:
scaler = StandardScaler()
scaler.set_output(transform='pandas')

df_scaled = scaler.fit_transform(df[['Open','High','Low','Close','Volume']])
df_scaled['Date'] = df['Date']
df_scaled.index = df['Date']
df_scaled

,Open,High,Low,Close,Volume,Date
Date,,,,,,
2021-01-11,-1.417733,-1.505471,-1.546049,-1.431805,-0.661957,2021-01-11
2021-01-12,-1.465901,-1.475692,-1.339428,-1.383091,-0.730572,2021-01-12
2021-01-13,-1.344805,-1.467609,-1.443461,-1.494502,-0.537809,2021-01-13
2021-01-14,-1.457798,-1.275745,-1.325461,-1.300548,-0.324247,2021-01-14
2021-01-15,-1.299337,-1.357425,-1.220946,-1.244166,-0.027828,2021-01-15
...,...,...,...,...,...,...
2021-07-02,-0.308956,-0.333865,-0.096331,-0.174260,-0.541020,2021-07-02
2021-07-05,-0.120333,-0.288770,-0.118004,-0.250939,-0.593095,2021-07-05
2021-07-06,-0.020845,0.521656,0.211915,0.506385,5.334342,2021-07-06


Now, let's create our training and test set:

In [ ]:
train = df_scaled.loc[:'06-2021']
test = df_scaled.loc['06-2021':]
train

,Open,High,Low,Close,Volume,Date
Date,,,,,,
2021-01-11,-1.417733,-1.505471,-1.546049,-1.431805,-0.661957,2021-01-11
2021-01-12,-1.465901,-1.475692,-1.339428,-1.383091,-0.730572,2021-01-12
2021-01-13,-1.344805,-1.467609,-1.443461,-1.494502,-0.537809,2021-01-13
2021-01-14,-1.457798,-1.275745,-1.325461,-1.300548,-0.324247,2021-01-14
2021-01-15,-1.299337,-1.357425,-1.220946,-1.244166,-0.027828,2021-01-15
...,...,...,...,...,...,...
2021-06-24,-0.299952,-0.395125,-0.263940,-0.433166,-0.362415,2021-06-24
2021-06-25,-0.349021,-0.350030,-0.171466,-0.203578,-0.541061,2021-06-25
2021-06-28,-0.164900,-0.216448,-0.048167,-0.069615,-0.588881,2021-06-28


We now create our independent and dependent data where we drop the 'Date' as it is not a feature to use in our inference step:

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error as mse, mean_absolute_percentage_error as mape

y_train = train['Open']
X_train = train.drop(['Open', 'Date'], axis=1)
y_test = test['Open']
X_test = test.drop(['Open', 'Date'], axis=1)

xgboost = GradientBoostingRegressor()
xgboost.fit(X_train, y_train)
predictions_train = xgboost.predict(X_train)
predictions_test = xgboost.predict(X_test)

print("MSE training: %.3f" % (mse(y_train, predictions_train) * 100))
print("MAPE training: %.3f" % (mape(y_train, predictions_train) * 100))

print("MSE test: %.3f" % (mse(y_test, predictions_test) * 100))
print("MAPE test: %.3f" % (mape(y_test, predictions_test) * 100))

MSE training: 0.168
MAPE training: 9.546
MSE test: 1.663
MAPE test: 125.990


We get decent error/percentage error on the training set, but the result on the test set is less impressive with over a 100% error rate.

Now, we are going to encode the date information into our XGBoost trees using one-hot encoded months and days:

In [ ]:
df_scaled['Month'] = df_scaled['Date'].dt.month
df_scaled['Month'] = df_scaled['Month'].astype('object')
df_scaled['Day'] = df_scaled['Date'].dt.day
df_scaled['Day'] = df_scaled['Day'].astype('object')
df_scaled = pd.get_dummies(df_scaled, drop_first=True)
df_scaled

,Open,High,Low,Close,Volume,Date,Month_2,Month_3,Month_4,Month_5,...,Day_22,Day_23,Day_24,Day_25,Day_26,Day_27,Day_28,Day_29,Day_30,Day_31
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-11,-1.417733,-1.505471,-1.546049,-1.431805,-0.661957,2021-01-11,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2021-01-12,-1.465901,-1.475692,-1.339428,-1.383091,-0.730572,2021-01-12,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2021-01-13,-1.344805,-1.467609,-1.443461,-1.494502,-0.537809,2021-01-13,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2021-01-14,-1.457798,-1.275745,-1.325461,-1.300548,-0.324247,2021-01-14,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2021-01-15,-1.299337,-1.357425,-1.220946,-1.244166,-0.027828,2021-01-15,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-07-02,-0.308956,-0.333865,-0.096331,-0.174260,-0.541020,2021-07-02,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2021-07-05,-0.120333,-0.288770,-0.118004,-0.250939,-0.593095,2021-07-05,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2021-07-06,-0.020845,0.521656,0.211915,0.506385,5.334342,2021-07-06,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
train = df_scaled.loc[:'06-2021']
test = df_scaled.loc['06-2021':]

y_train = train['Open']
X_train = train.drop(['Open', 'Date'], axis=1)
y_test = test['Open']
X_test = test.drop(['Open', 'Date'], axis=1)

xgboost = GradientBoostingRegressor()
xgboost.fit(X_train, y_train)
predictions_train = xgboost.predict(X_train)
predictions_test = xgboost.predict(X_test)

print("MSE training: %.3f" % (mse(y_train, predictions_train) * 100))
print("MAPE training: %.3f" % (mape(y_train, predictions_train) * 100))

print("MSE test: %.3f" % (mse(y_test, predictions_test) * 100))
print("MAPE test: %.3f" % (mape(y_test, predictions_test) * 100))

MSE training: 0.201
MAPE training: 11.250
MSE test: 1.523
MAPE test: 124.889


Adding these features does not improve predictive performance. We could consider hyperparameter tuning, finding more data, or using more specific machine learning models such as recurrent neural networks (e.g. LSTMs).